# Phase III: Explainable Boosting Machine

This notebook trains an InterpretML Explainable Boosting Classifier (EBM) to predict payment difficulty. It follows the shared preprocessing and validation workflow used by the decision-tree and logistic-regression notebooks. The model is configured with `interactions=0` to keep the explanation additive.

In [ ]:
# Dependencies can be installed with the following command (uncomment to run):
# %pip install -r requirements.txt

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from interpret import show
from interpret.data import ClassHistogram
from interpret.glassbox import ExplainableBoostingClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    recall_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_curve,
    roc_auc_score,
)
from sklearn.model_selection import GridSearchCV, StratifiedKFold

from common import (
    CATEGORICAL_COLUMNS,
    DATA_DICTIONARY,
    create_submission,
    encode_features,
    load_data,
    save_predictions,
    split_data,
)

train_data, test_data = load_data()

In [ ]:
print("Null values:")
print(train_data.isnull().sum())

print("\nData types:")
print(train_data.dtypes)

print("\nDescriptive statistics:")
display(train_data.describe().T)

target_column = next(
    column
    for column, role in DATA_DICTIONARY.items()
    if role == "target"
)
print("\nTarget counts:")
print(train_data[target_column].value_counts().sort_index())

In [ ]:
data_dictionary = DATA_DICTIONARY
categorical_columns = list(CATEGORICAL_COLUMNS)
split = split_data(train_data, test_data, data_dictionary)

predictor_columns = split.predictor_columns
target_column = split.target_column
X = split.X
y = split.y
X_train = split.X_train
X_validation = split.X_validation
y_train = split.y_train
y_validation = split.y_validation
X_test = split.X_test

In [ ]:
encoded = encode_features(split, categorical_columns)
encoder = encoded.encoder
numerical_columns = encoded.numerical_columns
X_train_encoded = encoded.X_train
X_validation_encoded = encoded.X_validation
X_test_encoded = encoded.X_test

X_train_encoded.head()

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=411)
cv_scores = {
    "accuracy": [],
    "balanced_accuracy": [],
    "recall_class_0": [],
    "recall_class_1": [],
}

for train_indices, validation_indices in cv.split(X_train_encoded, y_train):
    fold_ebm = ExplainableBoostingClassifier(
        interactions=0,
        outer_bags=4,
        max_rounds=5000,
        early_stopping_rounds=50,
        n_jobs=1,
        random_state=411,
    )
    fold_ebm.fit(
        X_train_encoded.iloc[train_indices],
        y_train.iloc[train_indices],
    )
    fold_predictions = fold_ebm.predict(
        X_train_encoded.iloc[validation_indices]
    )
    fold_targets = y_train.iloc[validation_indices]
    cv_scores["accuracy"].append(
        accuracy_score(fold_targets, fold_predictions)
    )
    cv_scores["balanced_accuracy"].append(
        balanced_accuracy_score(fold_targets, fold_predictions)
    )
    cv_scores["recall_class_0"].append(
        recall_score(fold_targets, fold_predictions, pos_label=0)
    )
    cv_scores["recall_class_1"].append(
        recall_score(fold_targets, fold_predictions, pos_label=1)
    )

for metric_name, scores in cv_scores.items():
    print(
        f"Five-fold {metric_name}: "
        f"{np.mean(scores):.3f} +/- {np.std(scores):.3f}"
    )

In [ ]:
histogram = ClassHistogram().explain_data(
    X_train,
    y_train,
    name="Training data",
)
show(histogram)

In [ ]:
ebm_baseline = ExplainableBoostingClassifier(
    interactions=0,
    outer_bags=4,
    max_rounds=5000,
    early_stopping_rounds=50,
    n_jobs=1,
    random_state=411,
)
ebm_baseline.fit(X_train_encoded, y_train)
baseline_predictions = ebm_baseline.predict(X_validation_encoded)

print("Baseline validation accuracy: "
      f"{accuracy_score(y_validation, baseline_predictions):.3f}")
print("Baseline validation balanced accuracy: "
      f"{balanced_accuracy_score(y_validation, baseline_predictions):.3f}")
print(classification_report(y_validation, baseline_predictions))

In [ ]:
param_grid = {
    "max_leaves": [2, 3],
    "learning_rate": [0.01, 0.03],
}

grid_search = GridSearchCV(
    estimator=ExplainableBoostingClassifier(
        interactions=0,
        outer_bags=4,
        max_rounds=5000,
        early_stopping_rounds=50,
        n_jobs=1,
        random_state=411,
    ),
    param_grid=param_grid,
    scoring="balanced_accuracy",
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=411),
    n_jobs=1,
)
grid_search.fit(X_train_encoded, y_train)
print(f"Best parameters: {grid_search.best_params_}")
print(f"Best CV balanced accuracy: {grid_search.best_score_:.3f}")

In [ ]:
ebm = ExplainableBoostingClassifier(
    interactions=0,
    outer_bags=4,
    max_rounds=5000,
    early_stopping_rounds=50,
    n_jobs=1,
    random_state=411,
    **grid_search.best_params_,
)
ebm.fit(X_train_encoded, y_train)

validation_predictions = ebm.predict(X_validation_encoded)
validation_probabilities = ebm.predict_proba(X_validation_encoded)[:, 1]
print(f"Validation accuracy: {accuracy_score(y_validation, validation_predictions):.3f}")
print("Validation balanced accuracy: "
      f"{balanced_accuracy_score(y_validation, validation_predictions):.3f}")
print(classification_report(y_validation, validation_predictions))

In [ ]:
ebm_global = ebm.explain_global(name="Optimized Explainable Boosting Classifier")
show(ebm_global)

term_importances = pd.Series(
    ebm.term_importances(),
    index=ebm.term_names_,
    name="importance",
)
top_importances = term_importances.nlargest(12).sort_values()
fig, ax = plt.subplots(figsize=(10, 6))
top_importances.plot.barh(ax=ax, color="#287271")
ax.set_title("Most influential terms in the optimized EBM")
ax.set_xlabel("Mean absolute contribution")
ax.set_ylabel("")
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
fig.savefig("../report/figures/ebm_global_importance.png", dpi=240, bbox_inches="tight")
plt.show()

In [ ]:
confusion = confusion_matrix(y_validation, validation_predictions)
confusion_display = ConfusionMatrixDisplay(
    confusion_matrix=confusion,
    display_labels=ebm.classes_,
)
confusion_display.plot(cmap=plt.cm.Blues)
plt.title("Optimized EBM confusion matrix")
plt.grid(False)
plt.show()

false_positive_rate, true_positive_rate, _ = roc_curve(
    y_validation,
    validation_probabilities,
)
auc_score = roc_auc_score(y_validation, validation_probabilities)
plt.figure(figsize=(7, 5))
plt.plot(
    false_positive_rate,
    true_positive_rate,
    color="darkorange",
    lw=2,
    label=f"ROC curve (AUC = {auc_score:.3f})",
)
plt.plot([0, 1], [0, 1], color="navy", lw=2, linestyle="--")
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel("False positive rate")
plt.ylabel("True positive rate")
plt.title("Optimized EBM ROC curve")
plt.legend(loc="lower right")
plt.show()

In [ ]:
validation_targets = y_validation.to_numpy()
prediction_values = np.asarray(validation_predictions)
candidate_cases = (
    ("Class 0: correctly predicted", np.flatnonzero((validation_targets == 0) & (prediction_values == 0))),
    ("Class 1: correctly detected", np.flatnonzero((validation_targets == 1) & (prediction_values == 1))),
    ("Class 1: missed by the model", np.flatnonzero((validation_targets == 1) & (prediction_values == 0))),
)
local_positions = []
local_labels = []
for case_label, positions in candidate_cases:
    if positions.size:
        local_positions.append(int(positions[0]))
        local_labels.append(case_label)

if not local_positions:
    raise ValueError("No representative validation cases were found.")

local_X = X_validation_encoded.iloc[local_positions]
local_y = y_validation.iloc[local_positions]
ebm_local = ebm.explain_local(local_X, local_y, name="Representative validation cases")
local_summary = pd.DataFrame(
    {
        "case": local_labels,
        "validation_row": X_validation.index[local_positions],
        "actual_class": local_y.to_numpy(),
        "predicted_class": prediction_values[local_positions],
        "predicted_probability": ebm.predict_proba(local_X)[:, 1],
    }
)
show(ebm_local)
with pd.option_context("display.max_colwidth", None):
    display(local_summary)

In [ ]:
model_name = "explainable_boosting"
test_predictions = ebm.predict(X_test_encoded)
submission = create_submission(test_data, test_predictions)
output_path = save_predictions(submission, model_name)
print(f"Predictions saved to {output_path}")

class_proportions = (
    submission["predicted_class"]
    .value_counts(normalize=True)
    .sort_index()
)
ax = class_proportions.plot(
    kind="bar",
    color=["#2563eb", "#f97316"],
    title="Proportion of predicted classes in test data",
    ylabel="Proportion",
    xlabel="Predicted class",
    ylim=(0, 1),
)
ax.set_xticklabels(class_proportions.index, rotation=0)
ax.yaxis.set_major_formatter(
    plt.FuncFormatter(lambda value, _: f"{value:.0%}")
)
for index, proportion in enumerate(class_proportions):
    ax.text(index, proportion + 0.02, f"{proportion:.1%}", ha="center")
plt.tight_layout()
plt.show()

submission.head()